# United States - Crime Rates - 1960 - 2014

### Step 1. Import the necessary libraries

In [ ]:
from pathlib import Path
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

DATA_ROOT = Path(os.environ.get("PANDAS_DATA_ROOT", r"../pandas_exercises"))

def data_file(name):
    direct = DATA_ROOT / name
    if direct.is_file():
        return direct
    matches = list(DATA_ROOT.rglob(name)) if DATA_ROOT.exists() else []
    if len(matches) != 1:
        raise FileNotFoundError(f"Set DATA_ROOT to your extracted data folder. Expected one {name}; found {len(matches)}.")
    return matches[0]


### Step 2. Import the dataset from this [address](https://raw.githubusercontent.com/guipsamora/pandas_exercises/master/04_Apply/US_Crime_Rates/US_Crime_Rates_1960_2014.csv). 

### Step 3. Assign it to a variable called crime.

In [ ]:
crime = pd.read_csv(data_file('US_Crime_Rates_1960_2014.csv'))
crime.head()

### Step 4. What is the type of the columns?

In [ ]:
crime.dtypes

##### Have you noticed that the type of Year is int64. But pandas has a different type to work with Time Series. Let's see it now.

### Step 5. Convert the type of the column Year to datetime64

In [ ]:
crime['Year'] = pd.to_datetime(crime['Year'].astype(str), format='%Y')

### Step 6. Set the Year column as the index of the dataframe

In [ ]:
crime = crime.set_index('Year').sort_index()

### Step 7. Delete the Total column

In [ ]:
crime = crime.drop(columns='Total')

### Step 8. Group the year by decades and sum the values

#### Pay attention to the Population column number, summing this column is a mistake

In [ ]:
decade = (crime.index.year // 10) * 10
crime_cols = crime.columns.drop('Population')
decades = crime.groupby(decade)[crime_cols].sum()
decades['Population'] = crime.groupby(decade)['Population'].max()
decades.index.name = 'Decade'
decades

### Step 9. What is the most dangerous decade to live in the US?

In [ ]:
# Violent and Property are totals; do not also add their component columns.
annual = crime.assign(crimes=crime['Violent'] + crime['Property'])
risk = annual.groupby((annual.index.year // 10) * 10).agg(crimes=('crimes', 'sum'), population_years=('Population', 'sum'), years=('Population', 'size'))
risk['annual_rate_per_100k'] = risk['crimes'] / risk['population_years'] * 100000
display(risk)
risk.loc[[risk['annual_rate_per_100k'].idxmax()]]